In [1]:
!pip install -q sentence-transformers faiss-cpu pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 61.5 MB/s eta 0:00:00


In [2]:
COURSE_MODULES = [
    {
        "module_id": "M1",
        "title": "Supervised Learning",
        "difficulty": "Beginner",
        "prerequisites": [],
        "estimated_minutes": 25,
        "transcript": """
Supervised learning uses labelled data to train a model. The model learns
the relationship between input features and known output labels.

Classification and regression are the two main supervised learning tasks.
Classification predicts a category, such as whether an email is spam or not.
Regression predicts a continuous number, such as the price of a house.

Common supervised learning algorithms include linear regression, logistic
regression, decision trees, and support vector machines. The model is
trained by minimising the difference between its predictions and the true
labels in the training data.
""",
    },
    {
        "module_id": "M2",
        "title": "Unsupervised Learning",
        "difficulty": "Intermediate",
        "prerequisites": ["M1"],
        "estimated_minutes": 30,
        "transcript": """
Unsupervised learning works with unlabeled data. The objective is to find
hidden patterns or groups in the data without being told the correct
answer in advance.

Clustering is the most common unsupervised learning technique. K-Means is
a popular clustering algorithm that groups similar data points together
based on distance, by repeatedly assigning points to the nearest cluster
centre and then recomputing the centres.

Dimensionality reduction, such as PCA, is another unsupervised technique.
It compresses many features into fewer components while keeping as much
information as possible, which is useful for visualisation.
""",
    },
    {
        "module_id": "M3",
        "title": "Reinforcement Learning",
        "difficulty": "Advanced",
        "prerequisites": ["M1", "M2"],
        "estimated_minutes": 35,
        "transcript": """
Reinforcement learning trains an agent to make decisions by interacting
with an environment. The agent receives a reward or penalty after each
action, and learns an optimal strategy, called a policy, through trial
and error.

Unlike supervised learning, there are no labelled examples. Instead the
agent must explore different actions to discover which ones lead to the
highest long-term reward, balancing exploration of new actions against
exploitation of actions already known to work well.

Reinforcement learning is used in robotics, game playing, and recommendation
systems, where an agent repeatedly interacts with an environment over time.
""",
    },
]

print("Modules defined:", [m["title"] for m in COURSE_MODULES])


Modules defined: ['Supervised Learning', 'Unsupervised Learning', 'Reinforcement Learning']


In [3]:
import re

def chunk_text(text: str, chunk_size: int = 3, overlap: int = 1) -> list[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if s]
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(sentences), step):
        chunk = " ".join(sentences[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks


def build_tagged_chunks(modules):
    """Returns a flat list of {text, module_id, title, difficulty} dicts —
    one per chunk, each remembering which module it came from.
    """
    tagged = []
    for module in modules:
        for chunk in chunk_text(module["transcript"]):
            tagged.append({
                "text": chunk,
                "module_id": module["module_id"],
                "title": module["title"],
                "difficulty": module["difficulty"],
            })
    return tagged

tagged_chunks = build_tagged_chunks(COURSE_MODULES)
print("Total chunks across all modules:", len(tagged_chunks))
tagged_chunks[0]


Total chunks across all modules: 10


{'text': 'Supervised learning uses labelled data to train a model. The model learns\nthe relationship between input features and known output labels. Classification and regression are the two main supervised learning tasks.',
 'module_id': 'M1',
 'title': 'Supervised Learning',
 'difficulty': 'Beginner'}

In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd

class CourseKnowledgeBase:
    """A metadata-aware knowledge base: embeds every chunk, indexes it with FAISS,
    and can filter retrieval down to a single module.
    """

    def __init__(self, tagged_chunks: list[dict], modules: list[dict], model_name: str = "all-MiniLM-L6-v2"):
        self.chunks = tagged_chunks
        self.modules_by_id = {m["module_id"]: m for m in modules}
        self.model = SentenceTransformer(model_name)

        texts = [c["text"] for c in tagged_chunks]
        embeddings = np.array(self.model.encode(texts)).astype("float32")
        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(embeddings)

    def search(self, question: str, top_k: int = 3, module_id: str | None = None) -> list[dict]:
        """Search the whole knowledge base, or restrict to one module_id."""
        q_emb = np.array(self.model.encode([question])).astype("float32")
        # over-fetch then filter, so module filtering still returns top_k relevant results
        fetch_k = top_k * 5 if module_id else top_k
        _, indices = self.index.search(q_emb, min(fetch_k, len(self.chunks)))

        results = [self.chunks[i] for i in indices[0]]
        if module_id:
            results = [r for r in results if r["module_id"] == module_id]
        return results[:top_k]

    def get_context(self, question: str, top_k: int = 3, module_id: str | None = None) -> str:
        results = self.search(question, top_k, module_id)
        return "\n\n".join(f"[{r['title']}] {r['text']}" for r in results)

    def module_summary(self) -> pd.DataFrame:
        """A readable table of the knowledge base's structure."""
        rows = []
        for m in self.modules_by_id.values():
            n_chunks = sum(1 for c in self.chunks if c["module_id"] == m["module_id"])
            rows.append({
                "Module": m["title"],
                "Difficulty": m["difficulty"],
                "Prerequisites": ", ".join(m["prerequisites"]) or "None",
                "Est. Minutes": m["estimated_minutes"],
                "Chunks Indexed": n_chunks,
            })
        return pd.DataFrame(rows)


knowledge_base = CourseKnowledgeBase(tagged_chunks, COURSE_MODULES)
print("Indexed vectors:", knowledge_base.index.ntotal)
knowledge_base.module_summary()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed vectors: 10


,Module,Difficulty,Prerequisites,Est. Minutes,Chunks Indexed
0,Supervised Learning,Beginner,None,25,4
1,Unsupervised Learning,Intermediate,M1,30,3
2,Reinforcement Learning,Advanced,"M1, M2",35,3


In [5]:
question = "What is K-Means clustering?"

print("=== Search across the whole knowledge base ===")
for r in knowledge_base.search(question, top_k=3):
    print(f"[{r['title']}] {r['text']}\n")

print("=== Search restricted to Module M1 (Supervised Learning) only ===")
restricted = knowledge_base.search(question, top_k=3, module_id="M1")
if not restricted:
    print("(No matching chunks in this module — as expected, K-Means lives in Unsupervised Learning.)")
else:
    for r in restricted:
        print(f"[{r['title']}] {r['text']}\n")


=== Search across the whole knowledge base ===
[Unsupervised Learning] Clustering is the most common unsupervised learning technique. K-Means is
a popular clustering algorithm that groups similar data points together
based on distance, by repeatedly assigning points to the nearest cluster
centre and then recomputing the centres. Dimensionality reduction, such as PCA, is another unsupervised technique.

[Unsupervised Learning] Unsupervised learning works with unlabeled data. The objective is to find
hidden patterns or groups in the data without being told the correct
answer in advance. Clustering is the most common unsupervised learning technique.

[Unsupervised Learning] Dimensionality reduction, such as PCA, is another unsupervised technique. It compresses many features into fewer components while keeping as much
information as possible, which is useful for visualisation.

=== Search restricted to Module M1 (Supervised Learning) only ===
[Supervised Learning] Supervised learning uses 

In [6]:
!pip install -q gradio

import gradio as gr

def view_module(module_title: str):
    module = next(m for m in COURSE_MODULES if m["title"] == module_title)
    module_chunks = [c["text"] for c in tagged_chunks if c["module_id"] == module["module_id"]]

    info = (
        f"Difficulty: {module['difficulty']}\n"
        f"Prerequisites: {', '.join(module['prerequisites']) or 'None'}\n"
        f"Estimated time: {module['estimated_minutes']} minutes\n"
        f"Chunks indexed: {len(module_chunks)}"
    )
    chunks_preview = "\n\n---\n\n".join(module_chunks)
    return info, chunks_preview


def search_kb(question: str, module_filter: str, top_k: int):
    if not question.strip():
        return "Enter a question above."
    module_id = None
    if module_filter != "All modules":
        module_id = next(m["module_id"] for m in COURSE_MODULES if m["title"] == module_filter)

    results = knowledge_base.search(question, top_k=int(top_k), module_id=module_id)
    if not results:
        return "No matching chunks found for this module filter."
    return "\n\n---\n\n".join(f"[{r['title']}] {r['text']}" for r in results)


module_titles = [m["title"] for m in COURSE_MODULES]

with gr.Blocks(title="Course Knowledge Base Explorer") as kb_demo:
    gr.Markdown("# 📚 Course Knowledge Base Explorer (Day 4)")

    with gr.Tab("Browse by Module"):
        module_dropdown = gr.Dropdown(choices=module_titles, label="Module", value=module_titles[0])
        info_output = gr.Textbox(label="Module Info", lines=4)
        chunks_output = gr.Textbox(label="Indexed Chunks", lines=10)
        module_dropdown.change(view_module, inputs=module_dropdown, outputs=[info_output, chunks_output])
        kb_demo.load(view_module, inputs=module_dropdown, outputs=[info_output, chunks_output])

    with gr.Tab("Search the Knowledge Base"):
        question_input = gr.Textbox(label="Question", placeholder="e.g. What is K-Means clustering?")
        module_filter_dropdown = gr.Dropdown(choices=["All modules"] + module_titles, value="All modules", label="Restrict to module")
        top_k_slider = gr.Slider(1, 5, value=3, step=1, label="Number of chunks to retrieve")
        search_btn = gr.Button("Search")
        search_output = gr.Textbox(label="Matching Chunks", lines=10)
        search_btn.click(search_kb, inputs=[question_input, module_filter_dropdown, top_k_slider], outputs=search_output)

kb_demo.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3d86041a112fcd422d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
